In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline


csv_path = os.path.join(path, "Q3_data.csv")

df = pd.read_csv(csv_path)


In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# 2. Do we have missing values?
def check_missing_values(df):

  # Get missing values using pandas
  missing_values = df.isnull().sum()

  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])

  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)




In [ ]:
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)


In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder

# Identify categorical columns
categorical_cols = df.select_dtypes(include='object').columns


df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

df.head()



label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

df

In [ ]:
# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Target")

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder



categorical_cols = df.select_dtypes(include='object').columns

if len(categorical_cols) > 0:
    print(f"Found categorical columns: {list(categorical_cols)}")


    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)


    label_encoders = {}
    for col in df.select_dtypes(include='object').columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le
    print("Categorical variables encoded.")
else:
    print("No object-type categorical columns found. Skipping encoding.")

df.head()

In [ ]:
from sklearn.preprocessing import StandardScaler


numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop("Golden feature", axis=1)
y = df['Golden feature']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np


# Initialize the model
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []



for train_idx, val_idx in kfold.split(X_train):
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Scale features within the fold
    scaler_fold = StandardScaler()
    X_fold_train_scaled = scaler_fold.fit_transform(X_fold_train)
    X_fold_val_scaled = scaler_fold.transform(X_fold_val)

    # Train and predict
    model.fit(X_fold_train_scaled, y_fold_train)
    y_fold_pred = model.predict(X_fold_val_scaled)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)


print(f"MAE:  {mae_scores.mean():,.2f}")



final_scaler = StandardScaler()
X_train_final_scaled = final_scaler.fit_transform(X_train)
final_model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
final_model.fit(X_train_final_scaled, y_train)
print("model trained ")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

feature_importances = final_model.feature_importances_
feature_names = X_train.columns


importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
})

importance_df = importance_df.sort_values(by='Importance', ascending=False)

# Plotting feature importance
plt.figure(figsize=(12, 8))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='blue')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Feature')
plt.show()

In [ ]:
import matplotlib.pyplot as plt


X_test_scaled = final_scaler.transform(X_test)

y_pred = final_model.predict(X_test_scaled)

plt.figure(figsize=(10, 6))
plt.hist(y_pred, bins=30, edgecolor='black', color='blue')
plt.title('Times')
plt.xlabel('Predicted Delivery')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: